# Bank Customer Churn — Exploratory Data Analysis (EDA)
**Notebook:** `GX_01_churn_eda`  
**Course:** Big Data Analysis (NOVA IMS)  
**Framework:** Apache Spark & PySpark on Databricks  

---

## 1. Project Context & Analytical Objective

### 1.1 Customer Churn in Retail Banking
Customer churn represents the loss of existing bank clients over a given observation window. For a commercial bank, acquiring new customers is considerably more expensive than retaining existing relationships. Identifying churn drivers and predicting at-risk accounts enables proactive retention campaigns, financial advisory outreach, or product repositioning.

### 1.2 Target Definition
- **Target Variable:** `Exited`
- **Definition:** A binary indicator where:
  - `1` represents a customer who left the bank (churned).
  - `0` represents an active/retained customer.

### 1.3 Analytical Boundary: Correlation vs. Causation
This exploratory notebook investigates observational patterns, data quality, feature distributions, and group disparities. 
- **Important Distinction:** Statistical associations (e.g., a higher churn rate among older customers or specific geographies) reflect **descriptive correlations**, not direct **causal mechanisms**.
- Observational features may be confounded by unmeasured factors (e.g., customer service interactions, competitor interest rates, regional economic conditions). Causal conclusions should not be asserted from observational EDA alone.

### 1.4 EDA Objective
The goal of this notebook is to **understand customer characteristics, assess data quality, and explore associations with churn** before any modelling. This EDA is descriptive — it does not establish causal relationships or predict individual outcomes.


## 2. Loading and Initial Inspection

### Analytical Question
What does the dataset contain, and how is it structured?

### Why It Matters
Before any analysis, we need to confirm the data loads correctly, understand column types, and classify columns by their analytical role. Loading errors or wrong type assumptions would compromise everything that follows.

### What This Section Does
- Loads `churn.csv` using PySpark with appropriate settings.
- Reports dimensions, schema, and a small preview (excluding surnames).
- Classifies columns as identifiers, numerical, categorical, binary, or target.
- Provides a data dictionary with explicit flags for uncertain meanings.

In [0]:
# ─── Configuration ───────────────────────────────────────────────
# Confirmed path: churn.csv is in our project Git folder.
# On Databricks, workspace files are accessible via the /Workspace/ prefix.
CHURN_CSV_PATH = "/Workspace/Users/marianasofiafariamartins@gmail.com/spark-databricks-analytics/churn.csv"

# Load the CSV with header=True (first row = column names) and
# inferSchema=True (Spark guesses column types from the data).
raw_churn_df = spark.read.csv(
    CHURN_CSV_PATH,
    header=True,
    inferSchema=True,
    sep=","
)

# raw_churn_df is the original, unchanged DataFrame.
# All transformations in later sections use separate analysis DataFrames.
print(f"Loaded successfully from:\n  {CHURN_CSV_PATH}")

In [0]:
# ─── Dimensions, schema, and preview ───────────────────────────
row_count = raw_churn_df.count()
col_count = len(raw_churn_df.columns)
print(f"Rows: {row_count:,}")
print(f"Columns: {col_count}\n")

# Print the inferred schema
print("Inferred schema:")
raw_churn_df.printSchema()

# Show 5 rows — drop Surname to avoid displaying identifying information
print("Preview (Surname excluded):")
display(raw_churn_df.drop("Surname").limit(5))

In [0]:
# ─── Column classification ──────────────────────────────────────
# Classify each column by its analytical role based on the schema
# and domain knowledge. This guides which columns to include in
# numerical analysis, categorical analysis, or exclude entirely.

column_info = [
    ("RowNumber",       "integer", "Identifier",  "Sequential row index — no analytical value"),
    ("CustomerId",      "integer", "Identifier",  "Unique customer ID — no analytical value"),
    ("Surname",         "string",  "Identifier",  "Customer surname — PII, excluded from analysis"),
    ("CreditScore",     "integer", "Numerical",   "Credit score (typical range 300-850)"),
    ("Geography",       "string",  "Categorical", "Customer's country of residence"),
    ("Gender",          "string",  "Categorical", "Customer's gender"),
    ("Age",             "integer", "Numerical",   "Customer age in years"),
    ("Tenure",          "integer", "Numerical",   "Years as a customer"),
    ("Balance",         "double",  "Numerical",   "Account balance"),
    ("NumOfProducts",   "integer", "Categorical", "Number of bank products used (1-4, ordinal)"),
    ("HasCrCard",       "integer", "Binary",      "1 = has credit card, 0 = no credit card"),
    ("IsActiveMember",  "integer", "Binary",      "1 = active member, 0 = inactive"),
    ("EstimatedSalary", "double",  "Numerical",   "Estimated annual salary"),
    ("Exited",          "integer", "Target",      "1 = churned (left), 0 = stayed"),
]

classification_df = spark.createDataFrame(column_info, ["Column", "InferredType", "Role", "Description"])
display(classification_df)

### Data Dictionary

| Column | Type | Role | Description | Uncertain? |
|--------|------|------|-------------|------------|
| `RowNumber` | integer | Identifier | Sequential row index | No |
| `CustomerId` | integer | Identifier | Unique customer identifier | No |
| `Surname` | string | Identifier / PII | Customer surname — excluded from analysis | No |
| `CreditScore` | integer | Numerical | Credit score (typically 300-850) | Range assumed from common scoring scales |
| `Geography` | string | Categorical | Customer's country | No |
| `Gender` | string | Categorical | Customer's gender | No |
| `Age` | integer | Numerical | Age in years | No |
| `Tenure` | integer | Numerical | Years as a customer | Unit (years vs months) assumed from context |
| `Balance` | double | Numerical | Account balance | Currency unspecified |
| `NumOfProducts` | integer | Categorical (ordinal) | Number of products (1-4) | Whether "products" includes all bank products unclear |
| `HasCrCard` | integer | Binary | Credit card holder (0/1) | No |
| `IsActiveMember` | integer | Binary | Active membership status (0/1) | Definition of "active" is unclear |
| `EstimatedSalary` | double | Numerical | Estimated annual salary | Estimation method unspecified |
| `Exited` | integer | Target | Churn indicator (0 = stayed, 1 = left) | Verified against data |

**Notes:**
- `CustomerId` is inferred as `integer`. No leading zeros were observed in the data, so integer storage is safe. If leading zeros existed, we would cast to `string` to preserve them.
- Currency unit for `Balance` and `EstimatedSalary` is not specified in the dataset metadata.
- `Tenure` is assumed to be in years based on the observed range (0-10).

## 3. Data Quality

### Analytical Question
Are there missing values, duplicates, invalid categories, or suspicious numerical values?

### Why It Matters
Data quality issues can bias analysis and modelling. We need to identify problems before interpreting results — but we do not fix them yet. This section documents issues and proposed actions without modifying the data.

### What This Section Does
- Counts missing values (NULL, NaN, whitespace-only strings) per column.
- Checks for exact duplicate rows and repeated customer IDs.
- Inspects categorical, binary, and target value distributions.
- Checks numerical ranges for suspicious observations.
- Summarises all findings and proposed actions in a compact table.

> **Principle:** We do not automatically drop outliers, impute missing values, or delete records. Any data modification decisions belong in the modelling notebook.

In [0]:
# ─── Missing values per column ──────────────────────────────────
from pyspark.sql.functions import col, count, when, isnan, trim

total_rows = raw_churn_df.count()

# Compute null count for every column in one pass
null_counts_row = raw_churn_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in raw_churn_df.columns
]).collect()[0]

# Build a readable summary DataFrame
missing_data = [
    (c, int(null_counts_row[c]), round(null_counts_row[c] / total_rows * 100, 2))
    for c in raw_churn_df.columns
]

missing_summary = spark.createDataFrame(missing_data, ["column", "null_count", "null_pct"])
print(f"Total rows: {total_rows:,}\n")
display(missing_summary)

In [0]:
# ─── Whitespace-only strings, NaN, and duplicate checks ─────────

# 1. Whitespace-only or empty strings in string columns
string_cols = [f.name for f in raw_churn_df.schema.fields
               if f.dataType.simpleString() == "string"]

print("=== String columns: null/empty/whitespace/NaN-string ===")
for c in string_cols:
    ws_count = raw_churn_df.filter(
        col(c).isNull() | (trim(col(c)) == "") | (col(c) == "NaN") | (col(c) == "null")
    ).count()
    print(f"  {c}: {ws_count}")

# 2. NaN values in numeric columns (NaN is different from NULL in Spark)
numeric_cols = [f.name for f in raw_churn_df.schema.fields
                if f.dataType.simpleString() in ("int", "integer", "double", "bigint")]

print("\n=== Numeric columns: NaN values ===")
for c in numeric_cols:
    nan_count = raw_churn_df.filter(isnan(col(c))).count()
    print(f"  {c}: {nan_count}")

# 3. Exact duplicate rows
dup_rows = raw_churn_df.count() - raw_churn_df.distinct().count()
print(f"\n=== Duplicates ===")
print(f"Exact duplicate rows: {dup_rows}")

# 4. Repeated CustomerIds (investigate before suggesting removal)
from pyspark.sql.functions import count as spark_count
repeated_ids = raw_churn_df.groupBy("CustomerId") \
    .agg(spark_count("*").alias("n")) \
    .filter("n > 1") \
    .count()
print(f"CustomerIds appearing more than once: {repeated_ids}")

In [0]:
# ─── Categorical, binary, and target value inspection ──────────
from pyspark.sql.functions import col, count, desc

# Categorical columns
print("=== Geography ===")
raw_churn_df.groupBy("Geography").count().orderBy(desc("count")).show()

print("=== Gender ===")
raw_churn_df.groupBy("Gender").count().orderBy(desc("count")).show()

# Binary columns
print("=== HasCrCard ===")
raw_churn_df.groupBy("HasCrCard").count().orderBy("HasCrCard").show()

print("=== IsActiveMember ===")
raw_churn_df.groupBy("IsActiveMember").count().orderBy("IsActiveMember").show()

# NumOfProducts (ordinal, 1-4)
print("=== NumOfProducts ===")
raw_churn_df.groupBy("NumOfProducts").count().orderBy("NumOfProducts").show()

# Target
print("=== Exited (target) ===")
raw_churn_df.groupBy("Exited").count().orderBy("Exited").show()

In [0]:
# ─── Numerical ranges and suspicious observations ───────────────
from pyspark.sql.functions import col

# Summary statistics using Spark's summary() method
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
print("=== Summary statistics ===")
display(raw_churn_df.summary("count", "min", "25%", "50%", "75%", "max")
        .select("summary", *num_cols))

# Check for suspicious values
print("\n=== Suspicious value checks ===")
checks = [
    ("CreditScore < 300 (below typical minimum)", raw_churn_df.filter(col("CreditScore") < 300).count()),
    ("CreditScore > 850 (above typical maximum)", raw_churn_df.filter(col("CreditScore") > 850).count()),
    ("Age < 18 (below adult age)",               raw_churn_df.filter(col("Age") < 18).count()),
    ("Age > 100",                                 raw_churn_df.filter(col("Age") > 100).count()),
    ("Balance < 0 (negative balance)",           raw_churn_df.filter(col("Balance") < 0).count()),
    ("EstimatedSalary < 0",                      raw_churn_df.filter(col("EstimatedSalary") < 0).count()),
    ("Tenure < 0",                                raw_churn_df.filter(col("Tenure") < 0).count()),
    ("Tenure > 10",                               raw_churn_df.filter(col("Tenure") > 10).count()),
]
for label, n in checks:
    flag = " [!]" if n > 0 else " [ok]"
    print(f"  {label}: {n}{flag}")

### Data Quality Summary

> **Interpretation after execution:** Complete this table based on the executed outputs above.

| Issue | Columns affected | Severity | Proposed action |
|-------|----------------|----------|-----------------|
| Missing values (NULL) | _fill after execution_ | — | — |
| Whitespace/empty strings | _fill after execution_ | — | — |
| NaN in numeric columns | _fill after execution_ | — | — |
| Exact duplicate rows | _fill after execution_ | — | — |
| Repeated CustomerIds | _fill after execution_ | — | — |
| Suspicious numerical values | _fill after execution_ | — | — |

**Key principle:** We do not automatically drop outliers, impute missing values, or delete records. Any data modification decisions should be made in the modelling notebook after careful consideration and with proper train/test discipline.

   
## 4. Target Distribution

### Analytical Question
How many customers churned versus stayed, and is the target variable clean?

### Why It Matters
The target distribution tells us whether the dataset is balanced. A heavily imbalanced target means a model could achieve high accuracy by always predicting the majority class, while completely missing the minority class (churners). Understanding the balance guides later evaluation metric choices.

### What This Section Does
- Separates valid target values (0 and 1) from any missing or invalid values.
- Computes counts and percentages for each class.
- Plots the target distribution using a consistent green palette.
- Discusses class imbalance and why accuracy alone may be misleading.

### How to Interpret
- If one class is much smaller than the other, the dataset is **imbalanced**.
- In that case, metrics like precision, recall, or F1-score are more informative than accuracy.
- We **do not** apply any resampling here — that is a modelling decision.

In [0]:
# ─── Section 4: Target distribution ────────────────────────────
from pyspark.sql.functions import col, count, lit

# Register a temporary view for Spark SQL used in later sections.
# This view references the raw DataFrame and does not create a persistent table.
raw_churn_df.createOrReplaceTempView("churn_raw")

total_rows = raw_churn_df.count()

# Separate valid (0 or 1) and invalid/missing target observations
valid_target = raw_churn_df.filter(col("Exited").isin(0, 1))
invalid_count = raw_churn_df.filter(
    ~col("Exited").isin(0, 1) | col("Exited").isNull()
).count()
valid_count = total_rows - invalid_count

print(f"Total rows:             {total_rows:,}")
print(f"Valid target (0 or 1):  {valid_count:,}")
print(f"Missing/invalid target: {invalid_count:,}")

# Distribution among valid targets only
print("\n=== Exited distribution (valid observations) ===")
target_dist = (
    valid_target.groupBy("Exited")
    .agg(count("*").alias("n_customers"))
    .withColumn("percentage", (col("n_customers") / lit(valid_count) * 100))
    .orderBy("Exited")
)
display(target_dist)

In [0]:
# ─── Plot target distribution ──────────────────────────────────
import matplotlib.pyplot as plt

# Collect the 2-row summary to pandas for plotting
# (only 2 rows — safe to collect to the driver)
target_pd = target_dist.toPandas()

# Consistent green palette used throughout the notebook
GREEN_MED  = "#27ae60"   # medium green — majority class
GREEN_DARK = "#0b5345"   # dark green — minority class

labels = ["Stayed (0)", "Churned (1)"]
colors = [GREEN_MED, GREEN_DARK]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, target_pd["n_customers"], color=colors, edgecolor="white")
ax.set_ylabel("Number of Customers")
ax.set_title("Target Distribution (Exited)")

for bar, pct in zip(bars, target_pd["percentage"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{int(bar.get_height()):,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

   
### Interpretation after execution

> After running the two cells above, answer these questions based on the actual outputs:

1. **What percentage of customers churned?** Look at the `percentage` column in the target distribution table.
2. **Is the dataset balanced?** A common guideline: if the minority class is below ~20-30% of the data, the dataset is considered imbalanced.
3. **Why might accuracy be misleading?** Consider: if ~80% of customers stayed, a model that always predicts "stayed" achieves ~80% accuracy — but it never identifies any churner. Metrics such as precision, recall, F1-score, or AUC-ROC would be more informative.
4. **How many invalid target values exist?** Check the `Missing/invalid target` count printed above. If it is 0, all 10,000 rows have a valid target.

   
## 5. Univariate Analysis

### Analytical Question
What do individual variables look like — their ranges, central tendencies, spread, and category frequencies?

### Why It Matters
Understanding each variable in isolation builds intuition about the customer base and reveals data quality issues (e.g., unexpected values, skewed distributions) that affect downstream analysis and modelling.

### What This Section Does
- **Numerical variables** (CreditScore, Age, Tenure, Balance, EstimatedSalary): Computes valid count, mean, standard deviation, minimum, maximum, and approximate quartiles using **Spark SQL**.
- **Distribution plots**: Creates histograms for three key numerical variables from Spark-computed bins (only ~20 rows collected to the driver per plot).
- **Categorical and binary variables** (Geography, Gender, NumOfProducts, HasCrCard, IsActiveMember): Shows counts and percentages using the **PySpark DataFrame API**.
- Identifiers (RowNumber, CustomerId) and the target (Exited) are excluded from numerical analysis.

### How to Interpret
- Compare mean vs median: if they differ substantially, the distribution is skewed.
- Large gaps between min/max and the quartiles may indicate outliers — but not necessarily errors.
- For categorical variables, check whether all values are expected and whether any category is very small.

In [0]:
# ─── Section 5: Numerical summaries using Spark SQL ───────────
# Exclude identifiers (RowNumber, CustomerId) and the target (Exited)
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]

# Build a Spark SQL query that computes summary statistics for each
# numerical column using UNION ALL. This demonstrates Spark SQL while
# keeping the query readable.
parts = []
for c in num_cols:
    parts.append(f"""
    SELECT '{c}'            AS variable,
           COUNT({c})       AS valid_count,
           ROUND(AVG({c}), 2)         AS mean,
           ROUND(STDDEV({c}), 2)      AS stddev,
           MIN({c})                   AS min_val,
           ROUND(percentile({c}, 0.25), 2) AS q1,
           ROUND(percentile({c}, 0.50), 2) AS median,
           ROUND(percentile({c}, 0.75), 2) AS q3,
           MAX({c})                   AS max_val
    FROM churn_raw""")

num_summary = spark.sql(" UNION ALL ".join(parts))
display(num_summary)

In [0]:
# ─── Distribution plots for key numerical variables ───────────
# We compute histogram bins in Spark and collect only ~20 rows per
# variable to the driver for plotting. This avoids collecting the
# full dataset.
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, min as Fmin, max as Fmax, least, lit

# Green palette (consistent across the notebook)
GREEN_MED = "#27ae60"

# Focus on 3 key variables to avoid repetitive charts
plot_cols = ["Age", "CreditScore", "Balance"]
n_bins = 20

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, c in zip(axes, plot_cols):
    # Get min and max from Spark
    bounds = raw_churn_df.select(
        Fmin(col(c)).alias("lo"), Fmax(col(c)).alias("hi")
    ).collect()[0]
    lo, hi = bounds["lo"], bounds["hi"]
    step = (hi - lo) / n_bins if hi != lo else 1.0

    # Compute histogram bins in Spark; collect only ~20 rows to pandas
    hist_df = (
        raw_churn_df.filter(col(c).isNotNull())
        .select(
            least(((col(c) - lo) / step).cast("int"), lit(n_bins - 1)).alias("bin")
        )
        .groupBy("bin")
        .count()
        .orderBy("bin")
    )
    hist_pd = hist_df.toPandas()
    hist_pd["bin_centre"] = hist_pd["bin"] * step + lo + step / 2

    ax.bar(hist_pd["bin_centre"], hist_pd["count"],
           width=step * 0.9, color=GREEN_MED, edgecolor="white")
    ax.set_title(c)
    ax.set_xlabel(c)
    ax.set_ylabel("Count")

plt.suptitle("Distribution of Key Numerical Variables", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# ─── Categorical and binary variable frequencies ─────────────
# Use the PySpark DataFrame API (complements the Spark SQL approach
# above — showing both APIs without duplicating every analysis).
from pyspark.sql.functions import col, count, lit, desc

total_rows = raw_churn_df.count()

cat_binary_cols = ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember"]

for c in cat_binary_cols:
    print(f"=== {c} ===")
    freq = (
        raw_churn_df.groupBy(c)
        .agg(count("*").alias("n"))
        .withColumn("pct", (col("n") / lit(total_rows) * 100))
        .orderBy(desc("n"))
    )
    freq.show(truncate=False)

   
## 6. Churn Rates by Customer Characteristics

### Analytical Question
Which customer groups have higher or lower churn rates, and are the differences meaningful given group sizes?

### Why It Matters
Comparing churn rates across groups helps identify segments where retention efforts might be focused. However, small groups can produce unstable rates, and associations in observational data do not imply causation.

### What This Section Does
- **Categorical/binary fields** (Geography, Gender, IsActiveMember, HasCrCard, NumOfProducts): Computes total customers, valid target count, churned count, and churn rate using **Spark SQL**.
- **Numerical fields** (Age, CreditScore, Balance, Tenure, EstimatedSalary): Compares descriptive statistics between customers who stayed and left using the **PySpark DataFrame API**.
- **Banded analysis**: Creates documented bands for Age, CreditScore, and Balance, then calculates churn rates per band.

### How to Interpret
- **Churn rate** = (churned customers / valid target customers) × 100. The denominator is valid target observations, not total rows.
- Compare **rates** (percentages), not just raw counts — a large group will naturally have more churned customers even if its rate is lower.
- Flag groups where `n < 100` — rates from very small groups are less reliable.
- Band boundaries (e.g., Age < 30, Balance = 0) are **analytical choices**, not established business thresholds.

In [0]:
# ─── Section 6: Churn rates by categorical fields (Spark SQL) ──
# For each categorical/binary field, compute:
#   total_customers  — all rows in the group
#   valid_target     — rows where Exited is 0 or 1
#   churned          — rows where Exited = 1
#   churn_rate_pct   — churned / valid_target × 100
# The denominator is valid target observations, so invalid/missing
# targets do not dilute the rate.

cat_fields = ["Geography", "Gender", "IsActiveMember", "HasCrCard", "NumOfProducts"]

for field in cat_fields:
    print(f"=== Churn rate by {field} ===")
    spark.sql(f"""
        SELECT
            {field}                                           AS group_value,
            COUNT(*)                                           AS total_customers,
            SUM(CASE WHEN Exited IN (0, 1) THEN 1 ELSE 0 END) AS valid_target,
            SUM(CASE WHEN Exited = 1 THEN 1 ELSE 0 END)        AS churned,
            ROUND(
                SUM(CASE WHEN Exited = 1 THEN 1 ELSE 0 END) * 100.0 /
                NULLIF(SUM(CASE WHEN Exited IN (0, 1) THEN 1 ELSE 0 END), 0),
                2
            ) AS churn_rate_pct
        FROM churn_raw
        GROUP BY {field}
        ORDER BY churn_rate_pct DESC
    """).show(truncate=False)

In [0]:
# ─── Plot churn rates by key categorical fields ───────────────
import matplotlib.pyplot as plt

# Green palette (consistent across the notebook)
GREEN_MED = "#27ae60"

# Collect churn rates for 3 key fields and plot
fields_to_plot = ["Geography", "Gender", "NumOfProducts"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, field in zip(axes, fields_to_plot):
    result = spark.sql(f"""
        SELECT {field} AS group_value,
               COUNT(*) AS n,
               ROUND(
                   SUM(CASE WHEN Exited = 1 THEN 1 ELSE 0 END) * 100.0 /
                   NULLIF(SUM(CASE WHEN Exited IN (0, 1) THEN 1 ELSE 0 END), 0),
                   2
               ) AS churn_rate_pct
        FROM churn_raw
        GROUP BY {field}
        ORDER BY group_value
    """).toPandas()

    result["group_value"] = result["group_value"].astype(str)
    # Convert Decimal (from SQL ROUND) to float for matplotlib
    result["churn_rate_pct"] = result["churn_rate_pct"].astype(float)
    result["n"] = result["n"].astype(int)

    ax.bar(result["group_value"], result["churn_rate_pct"],
           color=GREEN_MED, edgecolor="white")
    ax.set_title(f"Churn Rate by {field}")
    ax.set_ylabel("Churn Rate (%)")
    ax.set_xlabel(field)

    for i, (val, n) in enumerate(zip(result["churn_rate_pct"], result["n"])):
        ax.text(i, val + 0.3, f"{val:.1f}%\n(n={n:,})",
                ha="center", va="bottom", fontsize=8)

plt.suptitle("Churn Rate by Categorical Variables", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# ─── Descriptive statistics by Exited group (PySpark API) ─────
# Compare numerical variables between customers who stayed (0)
# and those who left (1). Uses the valid_target DataFrame from
# Section 4 to exclude any invalid target values.
from pyspark.sql.functions import (
    col, count, mean, stddev, min as Fmin, max as Fmax, round as Fround
)

num_cols = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]

for c in num_cols:
    print(f"--- {c} by Exited ---")
    valid_target.groupBy("Exited").agg(
        count(c).alias("n"),
        Fround(mean(c), 2).alias("mean"),
        Fround(stddev(c), 2).alias("stddev"),
        Fmin(c).alias("min"),
        Fmax(c).alias("max")
    ).orderBy("Exited").show()

In [0]:
# ─── Banded numerical fields: churn rates ───────────────────
# Create an analysis DataFrame with bands for Age, CreditScore,
# and Balance. The raw DataFrame remains unchanged.
# Band boundaries are analytical choices, not business thresholds.
from pyspark.sql.functions import when, col, count, sum as Fsum, round as Fround

analysis_df = (
    valid_target
    .withColumn("AgeBand",
        when(col("Age") < 30, "1. <30")
        .when(col("Age") < 40, "2. 30-39")
        .when(col("Age") < 50, "3. 40-49")
        .when(col("Age") < 60, "4. 50-59")
        .otherwise("5. 60+")
    )
    .withColumn("CreditScoreBand",
        when(col("CreditScore") < 500, "1. <500")
        .when(col("CreditScore") < 600, "2. 500-599")
        .when(col("CreditScore") < 700, "3. 600-699")
        .when(col("CreditScore") < 800, "4. 700-799")
        .otherwise("5. 800+")
    )
    .withColumn("BalanceBand",
        when(col("Balance") == 0, "1. Zero")
        .when(col("Balance") < 50000, "2. 1-50k")
        .when(col("Balance") < 100000, "3. 50-100k")
        .when(col("Balance") < 150000, "4. 100-150k")
        .otherwise("5. 150k+")
    )
)

# Helper: compute churn rate by band
def churn_by_band(df, band_col):
    return (
        df.groupBy(band_col)
        .agg(
            count("*").alias("n_customers"),
            Fsum(when(col("Exited") == 1, 1).otherwise(0)).alias("churned")
        )
        .withColumn(
            "churn_rate_pct",
            Fround(col("churned") / col("n_customers") * 100, 2)
        )
        .orderBy(band_col)
    )

print("=== Churn rate by Age band ===")
churn_by_band(analysis_df, "AgeBand").show(truncate=False)

print("=== Churn rate by CreditScore band ===")
churn_by_band(analysis_df, "CreditScoreBand").show(truncate=False)

print("=== Churn rate by Balance band ===")
churn_by_band(analysis_df, "BalanceBand").show(truncate=False)

   
### Interpretation after execution

> After running the cells above, answer these questions based on the actual outputs:

1. **Which geography has the highest churn rate?** Look at the churn rate table for Geography.
2. **Is there a gender difference in churn?** Compare the churn rates for Male and Female. Note any difference but remember: correlation is not causation.
3. **Does activity status matter?** Compare churn rates for active vs inactive members.
4. **Does number of products matter?** Look at NumOfProducts — do customers with more products churn more or less? Flag any group with n < 100.
5. **Which age band has the highest churn rate?** Check the banded analysis. Are any bands very small (n < 100)?
6. **Do zero-balance customers churn differently?** Compare the "Zero" balance band with other bands.
7. **Do mean Age, Balance, or CreditScore differ notably between stayed and churned groups?** Look at the descriptive statistics tables.

**Important:** Band boundaries (e.g., Age < 30, Balance = 0) are analytical choices made for this EDA, not established business thresholds. Different boundaries could reveal different patterns.

   
## 7. Selected Feature Relationships

### Analytical Question
Are any numerical features correlated with each other, and do combinations of categorical features reveal interesting churn patterns?

### Why It Matters
Understanding feature relationships helps identify potential predictors and multicollinearity issues. However, **correlation does not establish causation** — observed associations may be confounded by unmeasured factors.

### What This Section Does
- Computes **pairwise Pearson correlations** among numerical features using Spark SQL's `corr()` function.
- Excludes identifiers (RowNumber, CustomerId) and the target (Exited) from correlation calculations.
- Examines a **cross-tabulation** of IsActiveMember × NumOfProducts with churn rates and group sizes.
- Reports group sizes to flag small combinations.

### How to Interpret
- Correlation values range from -1 to +1. Values near 0 indicate no linear relationship.
- Spark's `corr()` drops rows with NULL in either column (row-wise pairwise deletion).
- Strong correlations between **features** (not with the target) may indicate multicollinearity.
- Observed associations are **descriptive** and do not prove predictive performance.

In [0]:
# ─── Section 7: Pairwise Pearson correlations ────────────────
# Exclude identifiers (RowNumber, CustomerId) and target (Exited).
# Spark SQL's corr() function drops rows with NULL in either column
# (row-wise pairwise deletion). NaN values are treated as NULL.

corr_cols = ["CreditScore", "Age", "Tenure", "Balance",
             "NumOfProducts", "EstimatedSalary"]

print("=== Pairwise Pearson correlations ===")
print("(NULL/NaN rows dropped row-wise by Spark's corr function)\n")

for i, c1 in enumerate(corr_cols):
    for c2 in corr_cols[i + 1:]:
        r = spark.sql(f"SELECT corr({c1}, {c2}) AS r FROM churn_raw").collect()[0]["r"]
        print(f"  {c1:>15s}  vs  {c2:<15s}  r = {r:+.4f}")

In [0]:
# ─── Cross-tabulation: IsActiveMember × NumOfProducts ───────
# Show group sizes and churn rates for each combination.
# This helps identify whether activity status moderates the
# relationship between number of products and churn.
from pyspark.sql.functions import col, count, sum as Fsum, round as Fround, when

cross_tab = (
    valid_target.groupBy("IsActiveMember", "NumOfProducts")
    .agg(
        count("*").alias("n_customers"),
        Fsum(when(col("Exited") == 1, 1).otherwise(0)).alias("churned")
    )
    .withColumn(
        "churn_rate_pct",
        Fround(col("churned") / col("n_customers") * 100, 2)
    )
    .orderBy("IsActiveMember", "NumOfProducts")
)

print("=== IsActiveMember × NumOfProducts: group sizes & churn rates ===")
cross_tab.show(truncate=False)

# Flag small groups (n < 100) where rates are less reliable
print("=== Small groups (n < 100) ===")
small_groups = cross_tab.filter(col("n_customers") < 100)
if small_groups.count() > 0:
    small_groups.show(truncate=False)
else:
    print("  No groups with n < 100.")

   
### Interpretation after execution

> After running the cells above, answer these questions based on the actual outputs:

1. **Which pair of numerical features has the strongest correlation?** Is the correlation positive or negative? Is it strong enough to suggest multicollinearity (|r| > 0.7)?
2. **Is Balance correlated with NumOfProducts?** A negative correlation might indicate that customers with more products have lower balances (or vice versa).
3. **In the cross-tab, which IsActiveMember × NumOfProducts combination has the highest churn rate?** Is that group large enough (n >= 100) for the rate to be reliable?
4. **Are any combinations very small (n < 100)?** Flag them and interpret their churn rates with caution.
5. **Reminder:** Correlation does not establish causation. These associations may be confounded by unmeasured factors.

   
## 8. Implications for Future Modelling

### Analytical Question
What preprocessing and modelling decisions does this EDA suggest — and what pitfalls should we avoid?

### Why It Matters
The findings from this EDA inform how we prepare data for modelling, but they do not guarantee model performance. Careful preprocessing discipline (train/test separation, fitting only on training data) is essential to avoid data leakage and over-optimistic evaluation.

### Columns to Exclude as Model Inputs
- **RowNumber**: A sequential index with no predictive meaning. Including it would let the model memorise row order.
- **CustomerId**: A unique identifier. It carries no generalisable signal and could cause overfitting.
- **Surname**: Personal identifiable information (PII). It has no analytical value for churn prediction and raises privacy concerns.

### Columns Requiring Preprocessing

| Column | Issue | Suggested treatment |
|--------|------|---------------------|
| Geography | Categorical (3 values) | One-hot or ordinal encoding |
| Gender | Categorical (2 values) | Binary encoding (0/1) |
| NumOfProducts | Categorical/ordinal (1-4) | Ordinal encoding or one-hot |
| HasCrCard | Binary (0/1) | Already numeric — no encoding needed |
| IsActiveMember | Binary (0/1) | Already numeric — no encoding needed |
| CreditScore | Numerical, varying scale | Scaling for distance-based models |
| Age | Numerical, varying scale | Scaling for distance-based models |
| Balance | Numerical, many zeros | Scaling; consider zero-indicator feature |
| EstimatedSalary | Numerical, varying scale | Scaling for distance-based models |
| Tenure | Numerical, bounded (0-10) | May not need scaling, depends on model |

### Critical Modelling Rules

1. **Exited must never be included among input features.** It is the target — including it would be data leakage.
2. **Learned preprocessing** (encoders, scalers, imputers) **must be fitted only on training data.** Fitting on the full dataset leaks information from the test set into the training process.
3. **The test set must remain untouched during model selection.** Any data-driven decision (feature selection, hyperparameter tuning, threshold selection) must use training data or cross-validation — not the test set.
4. **Data-driven feature decisions** (e.g., which bands to use, which interactions to include) must use **training data only**.

### Should Gender Be Included?

Gender is a sensitive attribute. Including it as a feature may improve predictions but raises fairness concerns:
- If Gender is a meaningful predictor, the model may learn different churn patterns for different genders — which could be legitimate (e.g., product preferences differ by gender) or could perpetuate bias.
- **Subgroup evaluation deserves consideration**: report performance metrics (precision, recall, F1) separately for each gender to check for disparities.
- Whether to include Gender is a modelling and ethical decision, not one that EDA alone can resolve.

### EDA Does Not Establish Model Performance

This EDA describes associations in the data. It does **not**:
- Prove that any feature will be a good predictor.
- Establish a baseline model accuracy or AUC.
- Determine which algorithm will perform best.
- Account for interactions that only become visible in a multivariate model.

   
## 9. Findings, Limitations and Next Steps

### Analytical Question
What have we learned from this EDA, what are its limitations, and what should the modelling notebook do next?

### Why It Matters
A clear summary ensures the EDA's findings are reproducible and actionable. Limitations prevent over-interpretation, and next steps provide a roadmap for the modelling phase.

### Findings Summary

> **Interpretation after execution:** Fill in the following based on the executed outputs from Sections 2-7. Do not invent figures — use only the actual results you see.

1. **Dataset**: _rows, columns, types_ (from Section 2 outputs).
2. **Data quality**: _missing values, duplicates, suspicious values_ (from Section 3 outputs).
3. **Target distribution**: _percentage churned, class imbalance status_ (from Section 4 outputs).
4. **Univariate insights**: _key distribution observations (skewness, outliers, zero balances)_ (from Section 5 outputs).
5. **Churn rate patterns**: _which groups have higher/lower churn_ (from Section 6 outputs).
6. **Correlations**: _notable pairwise correlations and cross-tab findings_ (from Section 7 outputs).

   
### Limitations

- **Provenance**: The dataset's source, collection method, and time period are not documented in the file itself. We rely on metadata assumptions (e.g., "Exited = 1 means churned").
- **Representativeness**: We do not know whether this sample is representative of the bank's full customer base or a specific segment.
- **Time coverage**: The dataset appears to be a single snapshot — we do not know the observation window (how long customers were tracked for churn).
- **Class imbalance**: If the churned class is a minority (as observed in Section 4), accuracy will be misleading. Metrics such as recall, F1, or AUC-ROC should be used for model evaluation.
- **Scale**: The dataset contains 10,000 rows — this does not require distributed processing on its own. We use Spark because it is the required framework for this project and because the same code scales to much larger datasets.

### Why Spark?

Apache Spark is the required framework for this Big Data Analytics project. While this dataset (10,000 rows) fits comfortably on a single machine, Spark provides:
- A **scalable implementation**: the same PySpark and Spark SQL code runs unchanged on datasets with billions of rows across a cluster.
- A **unified API**: DataFrame operations and SQL queries work identically whether the data is small or large.
- **Industry relevance**: Spark is widely used for big data processing in production environments.

### Next Steps for the Modelling Notebook

1. **Data preparation**: Apply the preprocessing decisions from Section 8 (encoding, scaling, feature selection).
2. **Train/test split**: Split the data before any fitting to avoid leakage. Consider stratified sampling to preserve class balance.
3. **Model selection**: Try baseline models (e.g., logistic regression, random forest) and compare using cross-validation on training data only.
4. **Evaluation**: Use appropriate metrics (precision, recall, F1, AUC-ROC) given the class imbalance. Report performance by subgroup (e.g., Gender) if fairness is a concern.
5. **Feature importance**: After training, examine which features the model considers most important and check for consistency with EDA findings.

---
*End of EDA notebook. Ensure all code cells have been executed and outputs are visible before submission.*